In [0]:
%run ../init_framework_gold

In [0]:
# --- 1. SOURCE INGESTION (Gold Layer) ---

# Read Loans Silver Stream
df_loans = read_delta_stream(
    spark=spark, 
    table_name=LOANS_SILVER
)

# Read Repayments Silver Stream
df_repayments = read_delta_stream(
    spark=spark, 
    table_name=REPAYMENTS_SILVER
)

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit

# Configured for 8-core cluster (16 shuffle partitions).
spark.conf.set("spark.sql.shuffle.partitions", "16")

# --- 2. STREAMING JOIN & SELECTION ---
# We use aliases 'p' for repayments and 'c' for loans (customers/contracts)
df_joined = df_repayments.alias("p").join(
    df_loans.alias("l"),
    "loan_id",
    "inner"
).select(
    col("l.member_id"),
    col("p.last_payment_amount"),
    col("l.monthly_installment"),
    col("p.total_payment_received"),
    col("l.funded_amount"),
    col("p._change_type").alias("repayments_change_type"), # Carried for CDF filtering in 
    col("l._change_type").alias("loans_change_type") # Carried for CDF filtering in     
)

In [0]:
# --- 3. APPLY GOLD METADATA ---
# # Injecting lineage and timestamp
df_with_metadata = apply_gold_metadata(df_joined, lit("calc_payment_performance"))

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# --- 4. SINK LOGIC (foreachBatch) ---
def upsert_payment_performance_to_gold(microBatchDF, batchId):
    """
    Stateless foreachBatch implementation for Gold Layer.
    Handles CDF metadata filtering, deduplication, scoring via lookup, and Delta Merge.
    """
    spark_session = microBatchDF.sparkSession

    # --- 1. Filter for 'Latest Truth' ---
    # Removes pre-images and accounts for State Store NULLs in the join
    df_clean = microBatchDF.filter(
        (col("repayments_change_type").isin("insert", "update_postimage"))
        & (
            col("loans_change_type").isin("insert", "update_postimage")
            | col("loans_change_type").isNull()
        )
    )

    # --- INJECTED: Micro-batch Deduplication ---
    # Partition by member_id, sort by newest update first to keep only the absolute latest state
    window_spec = Window.partitionBy("member_id").orderBy(col("_updated_at").desc())
    
    df_deduplicated = (df_clean
                       .withColumn("rn", row_number().over(window_spec))
                       .filter(col("rn") == 1)
                       .drop("rn"))

    # Register Deduplicated Source View for SQL transformations
    df_deduplicated.createOrReplaceTempView("batch_updates")

    # --- 2. Define Scoring Logic (Categorization) ---
    scoring_query = f""" 
        SELECT 
            member_id,
            CASE 
                WHEN last_payment_amount < (monthly_installment * 0.5) THEN 'VERY_BAD'
                WHEN last_payment_amount >= (monthly_installment * 0.5) AND last_payment_amount < monthly_installment THEN 'VERY_BAD'
                WHEN last_payment_amount = monthly_installment THEN 'GOOD'
                WHEN last_payment_amount > monthly_installment AND last_payment_amount <= (monthly_installment * 1.50) THEN 'VERY_GOOD'
                WHEN last_payment_amount > (monthly_installment * 1.50) THEN 'EXCELLENT'
                ELSE 'UNACCEPTABLE'
            END AS last_payment_key,
            CASE 
                WHEN total_payment_received >= (funded_amount * 0.50) THEN 'VERY_GOOD'
                WHEN total_payment_received < (funded_amount * 0.50) AND total_payment_received > 0 THEN 'GOOD'
                ELSE 'UNACCEPTABLE'
            END AS total_payment_key,
            _updated_at,
            _scoring_module
        FROM batch_updates         
    """

    # Map Keys to Points using Silver Lookup Table
    df_scored = spark_session.sql(scoring_query)
    df_scored.createOrReplaceTempView("scored_updates")

    # --- 3. Points Mapping Logic via Broadcast Join ---
    payments_query = f"""
        SELECT 
            perf.member_id,
            lpay.points AS last_payment_pts,
            tpay.points AS total_payment_pts,
            perf._updated_at,
            perf._scoring_module
        FROM scored_updates perf 
        JOIN lending_risk.silver.ref_score_points_mapping lpay
            ON perf.last_payment_key = lpay.rating_key
        JOIN lending_risk.silver.ref_score_points_mapping tpay
            ON perf.total_payment_key = tpay.rating_key
    """

    # Final Preparation for Merge
    df_payments = spark_session.sql(payments_query)
    df_payments.createOrReplaceTempView("payments_updates")

    # --- 4. Execute Idempotent Merge into Gold ---
    spark_session.sql(f"""
        MERGE INTO {CATALOG}.gold.calc_payment_performance AS target
        USING payments_updates AS source
        ON target.member_id = source.member_id
        WHEN MATCHED THEN
          UPDATE SET 
            target.last_payment_pts = source.last_payment_pts,
            target.total_payment_pts = source.total_payment_pts,
            target._updated_at = source._updated_at,
            target._scoring_module = source._scoring_module
        WHEN NOT MATCHED THEN
          INSERT (
            member_id, 
            last_payment_pts, 
            total_payment_pts, 
            _updated_at, 
            _scoring_module
          )
          VALUES (
            source.member_id, 
            source.last_payment_pts, 
            source.total_payment_pts, 
            source._updated_at, 
            source._scoring_module
          )
    """)

# --- STREAM TRIGGER ---
# Using availableNow=True for batch-like cost efficiency on a streaming engine
query = (
    df_with_metadata.writeStream
    .outputMode("append")
    .option("checkpointLocation", GOLD_CHECKPOINT_PAYMENT_PERF)
    .trigger(availableNow=True)
    .foreachBatch(upsert_payment_performance_to_gold)
    .start()
)

# Block notebook termination until micro-batch is committed
query.awaitTermination()

In [0]:
%sql
select count(1) from lending_risk.gold.calc_payment_performance 
limit 3;
-- 1168
-- 4501